# Решения: Практика: ключи, слияние и два указателя

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import math
import statistics
import time
import pandas as pd


def find_csv(name):
    for path in (Path(name), Path("../../data") / name, Path("../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден рядом с ноутбуком или в data/")


unsorted_df = pd.read_csv(find_csv("bank_transactions_unsorted.csv"))
by_id_df = pd.read_csv(find_csv("bank_transactions_sorted_by_txn_id.csv"))
by_amount_df = pd.read_csv(find_csv("bank_transactions_sorted_by_amount.csv"))
tiny_df = pd.read_csv(find_csv("bank_transactions_tiny.csv"))
COLS = ["txn_id", "amount", "day", "risk_score"]
unsorted_txns = list(unsorted_df[COLS].itertuples(index=False, name=None))
id_txns = list(by_id_df[COLS].itertuples(index=False, name=None))
amount_txns = list(by_amount_df[COLS].itertuples(index=False, name=None))
tiny_txns = list(tiny_df[COLS].itertuples(index=False, name=None))
id_list = [row[0] for row in id_txns]
amount_list = [row[1] for row in amount_txns]
assert id_list == sorted(id_list)
assert amount_list == sorted(amount_list)
print(f"Загружено {len(unsorted_txns)} транзакций; поля кортежа: {COLS}")


## Урок. 1. Merge rows

In [ ]:
def merge_rows_by_amount(left_rows, right_rows):
    i = j = 0; result = []
    while i < len(left_rows) and j < len(right_rows):
        if left_rows[i][1] <= right_rows[j][1]: result.append(left_rows[i]); i += 1
        else: result.append(right_rows[j]); j += 1
    return result + left_rows[i:] + right_rows[j:]

left_rows, right_rows = amount_txns[:40], amount_txns[40:80]
merged = merge_rows_by_amount(left_rows, right_rows)
assert merged == sorted(left_rows + right_rows, key=lambda r: r[1])


## Урок. 2–3. Gap и intersection

In [ ]:
def min_gap_between_sorted(left_values, right_values):
    i = j = 0; best = math.inf
    while i < len(left_values) and j < len(right_values):
        best = min(best, abs(left_values[i] - right_values[j]))
        if best == 0: return 0
        if left_values[i] < right_values[j]: i += 1
        else: j += 1
    return best

def intersect_sorted(left_values, right_values):
    i = j = 0; result = []
    while i < len(left_values) and j < len(right_values):
        if left_values[i] == right_values[j]: result.append(left_values[i]); i += 1; j += 1
        elif left_values[i] < right_values[j]: i += 1
        else: j += 1
    return result

window_a, window_b = amount_list[40:120], amount_list[400:480]
gap = min_gap_between_sorted(window_a, window_b)
assert gap == min(abs(a - b) for a in window_a for b in window_b)


## Урок. 4–6. Рейтинг и oracles

In [ ]:
top10 = sorted(unsorted_txns, key=lambda r: (-r[3], r[2], -r[1]))[:10]
toy_cases = [([1], [8]), ([1, 5, 10], [2, 9]), ([1, 2], [2, 3])]
gap_checks = [min_gap_between_sorted(a, b) == min(abs(x-y) for x in a for y in b) for a, b in toy_cases]
first_ids = sorted(r[0] for r in unsorted_txns[:500]); second_ids = sorted(r[0] for r in unsorted_txns[300:800])
shared_ids = intersect_sorted(first_ids, second_ids)
assert all(gap_checks) and shared_ids == sorted(set(first_ids) & set(second_ids))


## Урок. 7–9. Summary

In [ ]:
summary = {"merged_rows": len(merged), "min_gap": gap, "shared_ids": len(shared_ids), "top_risk": top10[0][0]}
MINI_REPORT = f"Слияние сохранило {len(merged)} строк в порядке amount. Минимальный разрыв сумм между окнами равен {gap}; oracle полного перебора дал то же значение. Пересечение окон содержит {len(shared_ids)} txn_id. Это описание структуры выборок, а не доказательство причины риска или поведения клиента."
acceptance = {"merge": len(merged) == 80, "gap_oracle": all(gap_checks), "intersection": shared_ids == sorted(set(first_ids) & set(second_ids)), "report": len(MINI_REPORT) >= 220}
assert set(acceptance.values()) == {True}


## ДЗ. Part A

In [ ]:
tiny_sorted = sorted(tiny_txns, key=lambda r: (r[2], -r[3], r[1]))
left_amounts = sorted(r[1] for r in tiny_txns[:40]); right_amounts = sorted(r[1] for r in tiny_txns[40:])
tiny_gap = min_gap_between_sorted(left_amounts, right_amounts)
a_ids = sorted(r[0] for r in unsorted_txns[:250]); b_ids = sorted(r[0] for r in unsorted_txns[150:400])
common = intersect_sorted(a_ids, b_ids)
assert common == sorted(set(a_ids) & set(b_ids))


## ДЗ. Challenge

In [ ]:
def audit_windows(rows_a, rows_b):
    left = sorted(rows_a, key=lambda r: r[1]); right = sorted(rows_b, key=lambda r: r[1])
    return {"merged_by_amount": merge_rows_by_amount(left, right), "min_amount_gap": min_gap_between_sorted([r[1] for r in left], [r[1] for r in right]), "shared_ids": intersect_sorted(sorted(r[0] for r in rows_a), sorted(r[0] for r in rows_b))}

report = audit_windows(unsorted_txns[:100], unsorted_txns[50:150])
OPS_NOTE = "После сортировки каждого окна слияние и пересечение выполняются за O(n + m); минимальный разрыв также требует одного прохода. Audit возвращает воспроизводимые факты о двух выборках. Ограничение: совпадение id и близость amount не объясняют риск и требуют бизнес-контекста."
assert len(OPS_NOTE) >= 240
